# K-Means Clustering on Well Log Data
## Machine learning for energy applications — Module 3, Exercise 1

In this exercise we apply K-Means clustering to well log data to propose groupings in
subsurface measurements. In petrophysics, clusters of log responses are used as *candidate*
rock types (electrofacies), fluid zones or completion intervals where no labels exist.
A cluster label is a hypothesis: it becomes a facies only when external evidence
(core, cuttings, depth trends, other wells) supports that name.

---
## 0 — Imports and Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples

# Plotting defaults
plt.rcParams.update({
    "figure.figsize": (8, 5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 12,
})

RANDOM_STATE = 42
print("Setup complete.")

---
## 1 — Load and Explore the Data

We load `w5.csv`, which contains well log measurements for a single well:
- **DEPTH** — measured depth
- **Pressure** — formation pressure
- **AI** — acoustic impedance
- **VpVs** — compressional-to-shear velocity ratio
- **SW** — water saturation
- **phid** — density porosity
- **vcl** — volume of clay

In [ ]:
# YOUR CODE HERE
# TODO: Load w5.csv into a DataFrame called df
# TODO: Print the shape and column names
# TODO: Display the first 10 rows with .head(10)


In [ ]:
# YOUR CODE HERE
# TODO: Display summary statistics using .describe()


In [ ]:
# YOUR CODE HERE
# TODO: Display data types using .info()
# TODO: Check for missing values using .isnull().sum()


---
## 2 — Visualize Feature Distributions

Before clustering it is essential to understand the distribution of each feature.
We plot histograms and a pairplot to look for natural groupings and correlations.

In [ ]:
# YOUR CODE HERE
# TODO: Define the list of features: ["Pressure", "AI", "VpVs", "SW", "phid", "vcl"]
# TODO: Create a 2x3 subplot figure with histograms for each feature
# Hint: Use ax.hist() with bins=40 for each feature


In [ ]:
# YOUR CODE HERE
# TODO: Create a pairplot of the features using sns.pairplot()
# Hint: Use diag_kind="kde" and set alpha and s in plot_kws


---
## 3 — Feature Selection and Standardization

K-Means is sensitive to feature scales because it uses Euclidean distance. We select
the petrophysical features (excluding DEPTH) and apply **StandardScaler** to give each
feature zero mean and unit variance.

> **Exploratory analysis of one fixed dataset.** The scaler is fitted on all samples of W5
> because we only describe this well: nothing is predicted for new data, so there is no
> training/validation/test split. If the clusters were used to label other wells, the scaler
> and K-Means would be fitted together (one pipeline) on the training wells only and applied
> to the new wells with `.predict`.

In [ ]:
# YOUR CODE HERE
# TODO: Define feature_cols = ["Pressure", "AI", "VpVs", "SW", "phid", "vcl"]
# TODO: Create X by selecting feature_cols from df and dropping NaN rows
# TODO: Fit a StandardScaler and transform X into X_scaled
# TODO: Print the number of samples, scaled means, and scaled stds to verify


---
## 4 — Apply K-Means with K=3

We start with K=3 clusters as an initial guess, to be checked with the diagnostics in
Sections 6–8. A geologist might expect sand, shale and transition zones, but that is a
hypothesis to test in Sections 9–10, not something the algorithm knows.

In [ ]:
# YOUR CODE HERE
# TODO: Create a KMeans model with n_clusters=3, random_state=RANDOM_STATE, n_init=10
# TODO: Fit and predict labels using .fit_predict(X_scaled)
# TODO: Print the unique labels and the number of samples in each cluster
# TODO: Print the inertia value


---
## 5 — Visualize Clusters in 2D

We project clusters onto pairs of features to inspect spatial coherence. AI vs VpVs
is a classic crossplot in rock physics.

In [ ]:
# YOUR CODE HERE
# TODO: Create a 1x3 subplot figure
# TODO: For each pair ("AI","VpVs"), ("phid","SW"), ("AI","vcl"):
#       scatter plot the data coloured by cluster label
# Hint: Loop through clusters and use a different color per cluster


---
## 6 — Elbow Method

The elbow method plots total within-cluster sum of squares (inertia) against the
number of clusters. The "elbow" point — where adding more clusters yields diminishing
returns — suggests candidate values of K. Inertia always falls as K grows, so it cannot
choose K on its own.

In [ ]:
# YOUR CODE HERE
# TODO: For K in range(1, 11), fit KMeans and record the inertia
# TODO: Plot inertia vs K with markers
# TODO: Print each K value with its corresponding inertia
# Hint: Use km.inertia_ to get the inertia after fitting


---
## 7 — Silhouette Analysis

The silhouette score measures how similar each point is to its own cluster compared
to neighbouring clusters. Values near +1 mean well inside the cluster, near 0 on a boundary;
negative values mean closer to another cluster on average. We evaluate K=2 through K=6.

In [ ]:
# YOUR CODE HERE
# TODO: For K in range(2, 7), fit KMeans and compute the silhouette_score
# TODO: Store scores in a list called sil_scores
# TODO: Plot silhouette score vs K
# TODO: Print each K value with its silhouette score


In [ ]:
# YOUR CODE HERE
# TODO: Create silhouette plots for each K value (K=2 to K=6)
# TODO: Create a 1x5 subplot figure
# TODO: For each K, compute silhouette_samples and plot a filled silhouette diagram
# TODO: Add a vertical dashed line at the average silhouette score
# Hint: Use ax.fill_betweenx() to create the silhouette bars for each cluster


---
## 8 — Choose K and Refit

We choose K with a stated rule: the highest mean silhouette for K = 2–6, checked against
the elbow plot. Mean silhouettes of about 0.3–0.4 indicate weakly separated groups, so the
chosen K is a working choice for this feature set, not evidence that the well contains that
many facies.

In [ ]:
# YOUR CODE HERE
# TODO: Select optimal_k as the K with the highest silhouette score from K_sil_range
#       (then check it against the elbow plot)
# TODO: Refit KMeans with optimal_k
# TODO: Print the chosen K, cluster sizes, inertia, and silhouette score
# Hint: Use np.argmax(sil_scores) to find the index of the best score


In [ ]:
# YOUR CODE HERE
# TODO: Visualize the optimal clusters with a 1x3 subplot figure
# TODO: Use scatter plots with pairs ("AI","VpVs"), ("phid","SW"), ("AI","vcl")
# TODO: Color points by cluster label using a colormap
# Hint: Use plt.colormaps["tab10"].resampled(optimal_k) for consistent cluster colors


---
## 9 — Interpret Clusters

We compute the mean feature values per cluster in original units and visualize them with a
heatmap in standardized units (for comparison across features). Read physical values from
the original-unit table, and write a hypothesis for what each cluster might represent:
a rock type, a fluid effect or a depth trend.

In [ ]:
# YOUR CODE HERE
# TODO: Add a "Cluster" column to X (use a copy) with the optimal labels
# TODO: Compute cluster means using .groupby("Cluster")[feature_cols].mean()
# TODO: Print the cluster means table


In [ ]:
# YOUR CODE HERE
# TODO: Transform the cluster means to standardized scale using the fitted scaler
# TODO: Create a heatmap with sns.heatmap() showing standardized cluster centroids
# Hint: Use cmap="RdBu_r", center=0, annot=True, fmt=".2f"


In [ ]:
# YOUR CODE HERE
# TODO: Create a 2x3 subplot figure with box plots for each feature grouped by cluster
# Hint: Use ax.boxplot() with patch_artist=True for coloured boxes and
#       tick_labels=[...] for the cluster names (Matplotlib >= 3.9)


---
## 10 — Clusters Along the Well Depth

Plotting cluster assignments against depth shows whether the clusters recur down the well,
as lithologies would, or mainly split it into depth intervals. DEPTH is not a feature, but
logs that trend with depth (for example Pressure, or compaction-sensitive AI and VpVs) can
carry depth into the clustering. Use the plot and the printed checks to write one hypothesis
per cluster and the evidence that would test it (core, cuttings, a rerun without Pressure).

In [ ]:
# YOUR CODE HERE
# TODO: Get the DEPTH values for the rows used in clustering (use X.index)
# TODO: Create a scatter plot of cluster assignments vs depth
# TODO: Invert the y-axis so depth increases downward
# TODO: Color points by cluster label
# Hint: Use fig size (4, 12) for a tall, narrow plot like a well log
# TODO: Print the correlation of each feature with DEPTH (X.corrwith(...))
# TODO: Print the depth range (count, min, median, max) of each cluster
# TODO: Write one hypothesis per cluster: lithology, fluid, or depth trend?


---
## Summary

In this exercise we covered:

1. **Data Loading and Exploration** — understanding well log feature distributions and correlations.
2. **Feature Standardization** — using StandardScaler before clustering (exploratory: fitted on this one well).
3. **K-Means with K=3** — applying clustering and visualizing results in 2D crossplots.
4. **Elbow Method** — plotting inertia vs K to identify diminishing returns.
5. **Silhouette Analysis** — quantifying cluster separation for K=2 through K=6.
6. **Choosing K** — a stated rule (highest mean silhouette, checked against the elbow); values of 0.3–0.4 mean weakly separated groups.
7. **Cluster Interpretation** — profiling clusters in original units, with heatmaps and box plots.
8. **Depth Integration** — mapping cluster assignments along the well to see whether they follow lithology, fluid or depth.

K-Means proposes candidate groupings of log responses. Treat each cluster label as a
hypothesis: check whether it follows lithology, fluid (SW) or depth (Pressure and compaction
trends), and confirm any electrofacies interpretation with core, cuttings or other wells.